# Phase 14.3 — Document Embeddings

This notebook generates vector embeddings for document chunks.

Input:
`genai_copilot.gold.document_chunks`

Output:
`genai_copilot.gold.document_embeddings`

The embeddings will be used by the RAG retrieval layer in Phase 14.4.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    ArrayType,
    FloatType,
    TimestampType
)

from datetime import datetime
import time

print("Imports successful.")

In [0]:
CHUNKS_TABLE = "genai_copilot.gold.document_chunks"
EMBEDDINGS_TABLE = "genai_copilot.gold.document_embeddings"

print("Input table:", CHUNKS_TABLE)
print("Output table:", EMBEDDINGS_TABLE)

In [0]:
print("Checking document chunks table...")

# Use the correct table name that exists in the schema
actual_table = "genai_copilot.gold.business_document_chunks"
spark.sql(f"""
DESCRIBE TABLE {actual_table}
""").show(truncate=False)

In [0]:
# Use the correct table name that exists in the schema
actual_table = "genai_copilot.gold.business_document_chunks"
chunks_df = spark.table(actual_table)

print("Total chunks:", chunks_df.count())

display(chunks_df.limit(10))

In [0]:
missing_chunks = (
    chunks_df
    .filter(
        F.col("chunk_text").isNull()
        | (F.length(F.trim(F.col("chunk_text"))) == 0)
    )
    .count()
)

print("Missing/empty chunks:", missing_chunks)

In [0]:
chunks_for_embedding = (
    chunks_df
    .select(
        "chunk_id",
        "document_id",
        "file_name",
        "document_type",
        "title",
        "chunk_index",
        "chunk_text",
        "created_at"
    )
    .filter(
        F.col("chunk_text").isNotNull()
        & (F.length(F.trim(F.col("chunk_text"))) > 0)
    )
)

print("Valid chunks for embedding:", chunks_for_embedding.count())

display(chunks_for_embedding.limit(10))

In [0]:
def generate_embedding(text):
    """
    Generate an embedding for one document chunk.

    The embedding provider is isolated inside this function
    so it can be replaced later without changing the RAG design.
    """

    if text is None or not str(text).strip():
        return None

    try:
        escaped_text = str(text).replace("'", "''")

        result = spark.sql(
            f"""
            SELECT ai_query(
                'databricks-gte-large-en',
                '{escaped_text}'
            ) AS embedding
            """
        ).collect()[0]["embedding"]

        return result

    except Exception as e:
        print("Embedding generation failed:")
        print(str(e))
        return None

In [0]:
test_row = (
    chunks_for_embedding
    .select("chunk_id", "chunk_text")
    .limit(1)
    .collect()
)

if not test_row:
    raise ValueError("No valid chunks found.")

test_chunk_id = test_row[0]["chunk_id"]
test_text = test_row[0]["chunk_text"]

print("Testing chunk:", test_chunk_id)
print("\nText:")
print(test_text[:500])

test_embedding = generate_embedding(test_text)

if test_embedding is None:
    raise RuntimeError(
        "Embedding generation failed. "
        "Do not continue until the embedding model works."
    )

print("\nEmbedding generated successfully.")
print("Embedding type:", type(test_embedding))
print("Embedding dimension:", len(test_embedding))

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {EMBEDDINGS_TABLE} (
    chunk_id STRING,
    document_id STRING,
    file_name STRING,
    document_type STRING,
    title STRING,
    chunk_index INT,
    chunk_text STRING,
    embedding ARRAY<FLOAT>,
    embedding_model STRING,
    created_at TIMESTAMP,
    embedding_created_at TIMESTAMP
)
USING DELTA
""")

print(f"Verified output table: {EMBEDDINGS_TABLE}")

In [0]:
rows = (
    chunks_for_embedding
    .select(
        "chunk_id",
        "document_id",
        "file_name",
        "document_type",
        "title",
        "chunk_index",
        "chunk_text",
        "created_at"
    )
    .collect()
)

print("Chunks to embed:", len(rows))

In [0]:
embedding_results = []

start_time = time.time()

for i, row in enumerate(rows):

    embedding = generate_embedding(row["chunk_text"])

    if embedding is not None:

        embedding_results.append({
            "chunk_id": row["chunk_id"],
            "document_id": row["document_id"],
            "file_name": row["file_name"],
            "document_type": row["document_type"],
            "title": row["title"],
            "chunk_index": row["chunk_index"],
            "chunk_text": row["chunk_text"],
            "embedding": [float(x) for x in embedding],
            "embedding_model": "databricks-gte-large-en",
            "created_at": row["created_at"],
            "embedding_created_at": datetime.now()
        })

    if (i + 1) % 10 == 0:
        print(
            f"Processed {i + 1}/{len(rows)} chunks"
        )

elapsed = time.time() - start_time

print("\nEmbedding generation complete.")
print("Input chunks:", len(rows))
print("Successful embeddings:", len(embedding_results))
print("Elapsed seconds:", round(elapsed, 2))

In [0]:
embedding_schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("file_name", StringType(), True),
    StructField("document_type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("chunk_index", IntegerType(), True),
    StructField("chunk_text", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False),
    StructField("embedding_model", StringType(), False),
    StructField("created_at", TimestampType(), True),
    StructField("embedding_created_at", TimestampType(), False)
])

embedding_rows = [
    (
        row["chunk_id"],
        row["document_id"],
        row["file_name"],
        row["document_type"],
        row["title"],
        row["chunk_index"],
        row["chunk_text"],
        row["embedding"],
        row["embedding_model"],
        row["created_at"],
        row["embedding_created_at"]
    )
    for row in embedding_results
]

embeddings_df = spark.createDataFrame(
    embedding_rows,
    schema=embedding_schema
)

print("Embedding DataFrame created.")

display(
    embeddings_df.select(
        "chunk_id",
        "document_id",
        "file_name",
        "title",
        "embedding_model"
    )
)

In [0]:
dimension_check = (
    embeddings_df
    .select(
        F.size("embedding").alias("embedding_dimension")
    )
    .groupBy("embedding_dimension")
    .count()
    .orderBy("embedding_dimension")
)

display(dimension_check)

In [0]:
(
    embeddings_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(EMBEDDINGS_TABLE)
)

print("Successfully saved:")
print(EMBEDDINGS_TABLE)

In [0]:
saved_df = spark.table(EMBEDDINGS_TABLE)

total_embeddings = saved_df.count()

print("Total embeddings:", total_embeddings)

display(
    saved_df.select(
        "chunk_id",
        "document_id",
        "file_name",
        "title",
        "embedding_model",
        F.size("embedding").alias("embedding_dimension")
    )
)

In [0]:
%sql
SELECT
    COUNT(*) AS total_embeddings
FROM genai_copilot.gold.document_embeddings;

In [0]:
%sql
SELECT
    embedding_model,
    COUNT(*) AS embedding_count
FROM genai_copilot.gold.document_embeddings
GROUP BY embedding_model;

In [0]:
%sql
SELECT
    SIZE(embedding) AS embedding_dimension,
    COUNT(*) AS record_count
FROM genai_copilot.gold.document_embeddings
GROUP BY SIZE(embedding);